# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maqsood-Ahmed110/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why
Method: Random Forest classifier.

My lane is scoring/ranking, and my w04 baseline was two hand-set flags
combined linearly (stale_visible, ctr_gap). A Random Forest fits because
it can learn non-linear interactions between staleness, CTR, position,
and volume that a fixed linear rule can't — e.g. "stale AND low-volume"
might matter differently than "stale AND high-volume," which my baseline
treats identically. Random Forest also gives permutation importance for
free, which lets me check whether the model actually leans on the same
signals I hand-picked, or finds something I missed.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

Grouped split by client_hash_id, using GroupShuffleSplit — not a random
row split. My rows aren't independent: many pages belong to the same
client, and a random split would let the model see other pages from the
same client in training, then "predict" a held-out page from that same
client — leaking client-specific patterns rather than testing whether
the model generalizes to genuinely unseen clients.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline
Recreating the exact w04 baseline rule on this same data, then training
a Random Forest on the same grouped train/test split, and comparing both
on the same held-out test rows using the same metrics.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

df = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_90d,
               SUM(gsc_clicks) AS clicks_90d,
               AVG(gsc_avg_position) AS avg_position,
               MAX(report_date) AS last_report_date
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT c.content_hash_id, a.client_hash_id, c.content_updated_date, c.word_count,
           a.impressions_90d, a.clicks_90d, a.avg_position,
           DATE_DIFF('day', c.content_updated_date, a.last_report_date) AS days_since_last_update
    FROM read_parquet('{REL}/dim_content.parquet') c
    JOIN agg a ON c.content_hash_id = a.content_hash_id
""").df()

df['ctr'] = df['clicks_90d'] / df['impressions_90d'].replace(0, pd.NA)
df['position_tier'] = pd.cut(df['avg_position'], bins=[0,3,10,20,999], labels=['1-3','4-10','11-20','21+'])

df['has_update_date'] = df['content_updated_date'].notna()
df['stale_visible_flag'] = (df['has_update_date'] & (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)).astype(int)
tier_avg_ctr = df.groupby('position_tier', observed=True)['ctr'].transform('mean')
df['ctr_gap_flag'] = (df['ctr'] < tier_avg_ctr * 0.5).astype(int)
df['baseline_score'] = 0.5 * df['stale_visible_flag'] + 0.5 * df['ctr_gap_flag']
df['baseline_action'] = (df['baseline_score'] > 0).astype(int)

df['label'] = (df['impressions_90d'] < df['impressions_90d'].median()).astype(int)

feature_cols = ['impressions_90d', 'clicks_90d', 'avg_position', 'word_count', 'days_since_last_update']
model_df = df.dropna(subset=feature_cols + ['label', 'client_hash_id'])

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train, test = model_df.iloc[train_idx], model_df.iloc[test_idx]

X_tr, y_tr = train[feature_cols], train['label']
X_te, y_te = test[feature_cols], test['label']

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
rf_pred = rf.predict(X_te)
baseline_pred = test['baseline_action']

print("=== BASELINE (w04 rule) on held-out test set ===")
print(classification_report(y_te, baseline_pred, digits=3))

print("\n=== MODEL (Random Forest) on same held-out test set ===")
print(classification_report(y_te, rf_pred, digits=3))

summary = pd.DataFrame({
    'baseline': [precision_score(y_te, baseline_pred), recall_score(y_te, baseline_pred), f1_score(y_te, baseline_pred)],
    'model':    [precision_score(y_te, rf_pred), recall_score(y_te, rf_pred), f1_score(y_te, rf_pred)],
}, index=['precision', 'recall', 'f1'])
print("\n=== Model vs Baseline summary ===")
print(summary)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== BASELINE (w04 rule) on held-out test set ===
              precision    recall  f1-score   support

           0      0.986     0.319     0.483     21713
           1      0.053     0.894     0.100       928

    accuracy                          0.343     22641
   macro avg      0.520     0.607     0.291     22641
weighted avg      0.948     0.343     0.467     22641


=== MODEL (Random Forest) on same held-out test set ===
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     21713
           1      1.000     1.000     1.000       928

    accuracy                          1.000     22641
   macro avg      1.000     1.000     1.000     22641
weighted avg      1.000     1.000     1.000     22641


=== Model vs Baseline summary ===
           baseline  model
precision  0.053178    1.0
recall     0.894397    1.0
f1         0.100387    1.0


## 4. Errors and interpretation

Recreating the exact w04 baseline rule on this same data, then training
a Random Forest on the same grouped train/test split, and comparing both
on the same held-out test rows using the same metrics.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean
}).sort_values('importance_mean', ascending=False)
print("=== Permutation importance ===")
print(importance_df)

test_results = test.copy()
test_results['model_pred'] = rf_pred
test_results['baseline_pred'] = baseline_pred
test_results['true_label'] = y_te.values

disagree = test_results[test_results['model_pred'] != test_results['baseline_pred']]
print(f"\nRows where model and baseline disagree: {len(disagree)} of {len(test_results)}")
print(disagree[['impressions_90d','avg_position','days_since_last_update','model_pred','baseline_pred','true_label']].head(10))


=== Permutation importance ===
                  feature  importance_mean
0         impressions_90d         0.078398
1              clicks_90d         0.000000
2            avg_position         0.000000
3              word_count         0.000000
4  days_since_last_update         0.000000

Rows where model and baseline disagree: 14876 of 22641
     impressions_90d  avg_position  days_since_last_update  model_pred  \
729              9.0     12.000000                      34           0   
738              3.0     63.000000                      34           0   
740             16.0     50.820513                      34           0   
752             12.0     14.818182                      34           0   
813             95.0     32.950860                      34           0   
832             24.0     18.095238                      34           0   
881             10.0     50.500000                      34           0   
891             13.0     28.533333                      34     

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.